In [9]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = "0"

# Path to root of this project, contains lots of modules
import sys
sys.path.insert(0, os.path.abspath('../'))
sys.path.insert(0, os.getcwd())

import math
import random
import numpy
from matplotlib import pyplot
from matplotlib import cm
from sklearn.preprocessing import StandardScaler
from tslearn.clustering import TimeSeriesKMeans
import torch
from torch import nn, optim
from src.learning_shapelets import LearningShapelets

import pandas as pd
from sklearn.model_selection import StratifiedKFold

import numpy as np

In [10]:


# series = pd.read_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\RFMN\\sktime-tutorial-pydata-global-2021\\notebooks\\univariatetest_data.csv', header=0, parse_dates=[0], index_col=0, squeeze=True)
# data = pd.read_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\RFMN\\sktime-tutorial-pydata-global-2021\\notebooks\\test.csv')
series = pd.read_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\RFMN\\Reflex-Fuzzy-Network\\Arduino_train_test_labels.csv')

series = series.iloc[:,1:]


print(type(series))
print(series)


<class 'pandas.core.frame.DataFrame'>
       Displacement  Force    Work  Label
0                 4   0.00    0.00      1
1                 3   0.00    0.00      1
2                 0   0.00    0.00      1
3                 3   0.00    0.00      1
4                 2   0.00    0.00      1
...             ...    ...     ...    ...
10795            10  21.40  213.95      0
10796             9  21.44  192.93      0
10797             8  21.44  171.50      0
10798            10  21.44  214.37      0
10799             9  21.40  192.56      0

[10800 rows x 4 columns]


In [11]:
skf = StratifiedKFold(n_splits=10, shuffle= True, random_state=42)

target = series.loc[:,'Label']

print(target)



0        1
1        1
2        1
3        1
4        1
        ..
10795    0
10796    0
10797    0
10798    0
10799    0
Name: Label, Length: 10800, dtype: int64


In [12]:
fold_no = 1
for train_index, val_index in skf.split(series, target):
    train = series.loc[train_index,:]
    val = series.loc[val_index,:]
    train.to_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\RFMN\\Reflex-Fuzzy-Network\\cross_val_data_LS1\\' + 'train_fold_' + str(fold_no) + '.csv')
    val.to_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\RFMN\\Reflex-Fuzzy-Network\\cross_val_data_LS1\\' + 'val_fold_' + str(fold_no) + '.csv')
    fold_no += 1

In [13]:
def sample_ts_segments(X, shapelets_size, n_segments=10000):
    """
    Sample time series segments for k-Means.
    """
    n_ts, n_channels, len_ts = X.shape
    samples_i = random.choices(range(n_ts), k=n_segments)
    segments = numpy.empty((n_segments, n_channels, shapelets_size))
    for i, k in enumerate(samples_i):
        s = random.randint(0, len_ts - shapelets_size)
        segments[i] = X[k, :, s:s+shapelets_size]
    return segments


def get_weights_via_kmeans(X, shapelets_size, num_shapelets, n_segments=10000):
    """
    Get weights via k-Means for a block of shapelets.
    """
    segments = sample_ts_segments(X, shapelets_size, n_segments).transpose(0, 2, 1)
    k_means = TimeSeriesKMeans(n_clusters=num_shapelets, metric="euclidean", max_iter=50).fit(segments)
    clusters = k_means.cluster_centers_.transpose(0, 2, 1)
    return clusters


def eval_accuracy(model, X, Y):
    predictions = model.predict(X)
    if len(predictions.shape) == 2:
        predictions = predictions.argmax(axis=1)
        print(predictions)
    print(f"Accuracy: {(predictions == Y).sum() / Y.size}")

In [14]:
count = 1

for fold_no in range(1,11):
    newtrain = pd.read_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\RFMN\\Reflex-Fuzzy-Network\\cross_val_data_LS\\' + 'train_fold_' + str(fold_no) + '.csv')
    newval = pd.read_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\RFMN\\Reflex-Fuzzy-Network\\cross_val_data_LS\\' + 'val_fold_' + str(fold_no) + '.csv')



    new_array_train = newtrain['Work'].to_numpy() 
    y_train = newtrain['Label'].to_numpy() 

    new_array_test = newval['Work'].to_numpy() 
    y_test = newval['Label'].to_numpy() 


    y_train = y_train[~np.isnan(y_train)]
    y_test = y_test[~np.isnan(y_test)]

    X_train = new_array_train.reshape(y_train.size,1,1)
    np.unique(y_train)
    

    X_test= new_array_test.reshape(y_test.size,1,1)
    np.unique(y_test)

    n_ts, n_channels, len_ts = X_train.shape
    loss_func = nn.CrossEntropyLoss()
    num_classes = len(set(y_train))
    # learn 2 shapelets of length 130
    shapelets_size_and_len = {1: 3000}
    dist_measure = "euclidean"
    lr = 1e-2
    wd = 1e-3
    epsilon = 1e-7

    learning_shapelets = LearningShapelets(shapelets_size_and_len=shapelets_size_and_len,
                                       in_channels=n_channels,
                                       num_classes=num_classes,
                                       loss_func=loss_func,
                                       to_cuda=False,
                                       verbose=1,
                                       dist_measure=dist_measure)
    
    for i, (shapelets_size, num_shapelets) in enumerate(shapelets_size_and_len.items()):
        weights_block = get_weights_via_kmeans(X_train, shapelets_size, num_shapelets)
        learning_shapelets.set_shapelet_weights_of_block(i, weights_block)

    optimizer = optim.Adam(learning_shapelets.model.parameters(), lr=lr, weight_decay=wd, eps=epsilon)
    learning_shapelets.set_optimizer(optimizer)

    losses = learning_shapelets.fit(X_train, y_train, epochs=2000, batch_size=256, shuffle=False, drop_last=False)
    
    eval_accuracy(learning_shapelets, X_test, y_test)

    print("done with ", count)
    count +=1



Loss: 11345.9990234375: 100%|██████████| 2000/2000 [1:51:21<00:00,  3.34s/it]  


[7 7 7 ... 0 0 0]
Accuracy: 0.3074074074074074
done with  1


Loss: 11235.68359375: 100%|██████████| 2000/2000 [1:36:54<00:00,  2.91s/it]       


[7 7 7 ... 0 0 0]
Accuracy: 0.2833333333333333
done with  2


Loss: 745.7686767578125: 100%|██████████| 2000/2000 [1:41:06<00:00,  3.03s/it]  


...

[1 1 1 ... 0 1 1]
Accuracy: 0.43148148148148147
done with  8


Loss: 0.0: 100%|██████████| 2000/2000 [2:00:38<00:00,  3.62s/it]                


[1 1 1 ... 0 0 0]
Accuracy: 0.4287037037037037
done with  9


Loss: 1482.6953125: 100%|██████████| 2000/2000 [1:54:38<00:00,  3.44s/it]       


[1 1 1 ... 0 0 0]
Accuracy: 0.41759259259259257
done with  10
